In [1]:
import sys
from pathlib import Path

parent_folder = Path.cwd().parent
if str(parent_folder) not in sys.path:
    sys.path.append(str(parent_folder))
    
import warnings
warnings.filterwarnings('ignore')
import numpy as np

import simulation_functional as sim
import xobjects as xo
import copy

In [6]:
gemitt = 60e-6
n_turns=5000
n_particles = 250

gamx = 0.13112848113527537 # gamma at 0
gamy = 0.35349943817994744
Jx_max = 5*gemitt#gamx * 0.05**2
Jy_max = 0.1*gemitt
Jx = np.linspace(0, Jx_max, 50)
Jy = np.linspace(0, Jy_max, 5)
Jx, Jy = np.meshgrid(Jx,Jy)
Jx = Jx.flatten()
Jy = Jy.flatten()

x = np.sqrt(Jx / gamx)
y = np.sqrt(Jy / gamy)
p = np.zeros_like(x)

line = sim.build_line(folder=Path('../Lattice_Files/02_Aperture_Lattice_res/'))

s_list = np.linspace(0, line.get_length(), 13*2, endpoint=False)
strength_list = 5e-3 * np.cos(13 * 2*np.pi * s_list / line.get_length())
line = sim.add_dipole(line, strength=strength_list, s=s_list, mode='normal')

particles_base = line.build_particles(
    x=x+line.twiss(method='4d', freeze_longitudinal=True).x[0],
    y=y+line.twiss(method='4d', freeze_longitudinal=True).y[0],
    px=p+line.twiss(method='4d', freeze_longitudinal=True).px[0],
    py=p+line.twiss(method='4d', freeze_longitudinal=True).py[0],
                         method='4d', freeze_longitudinal=True)

Converting sequence "synchrotron":   0%|          | 0/1990 [00:00<?, ?it/s]

Slicing line:   0%|          | 0/3980 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.


In [7]:
for indirect in [False, True]:
    line = sim.build_line(folder=Path('../Lattice_Files/02_Aperture_Lattice_res/'))

    s_list = np.linspace(0, line.get_length(), 13*2, endpoint=False)
    strength_list = 5e-3 * np.cos(13 * 2*np.pi * s_list / line.get_length())
    line = sim.add_dipole(line, strength=strength_list, s=s_list, mode='normal')
    
    particles = copy.deepcopy(particles_base)
    
    line = sim.add_monitors(line, particles, n_monitors=1, n_turns=n_turns)
    
    twiss = line.twiss(method='4d', freeze_longitudinal=True)
    
    line, beam_df, aper_df = sim.add_spacecharge(line, beam_intensity=1e13,
                                                 n_interactions=100, 
                                                 gemitt=gemitt, indirect=indirect)
    
    line.build_tracker(_context=xo.ContextCpu(omp_num_threads=10))
    line.track(particles, num_turns=n_turns, time=True)
    
    sim.save(line, particles, twiss, filename=f'data/kick_True_ind_{indirect}_smally', aper_df=aper_df, beam_df=beam_df)

Converting sequence "synchrotron":   0%|          | 0/1990 [00:00<?, ?it/s]

Slicing line:   0%|          | 0/3980 [00:00<?, ?it/s]

Slicing line:   0%|          | 0/4126 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.


Slicing line:   0%|          | 0/4127 [00:00<?, ?it/s]

...

Slicing line:   0%|          | 0/3980 [00:00<?, ?it/s]

Slicing line:   0%|          | 0/4126 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.


Slicing line:   0%|          | 0/4127 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.
Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.
File saved at data/kick_True_ind_True_smally
